# 09 - Statistics, figures and tables for the paper
Per-instance paired bootstrap 95% CIs, Wilcoxon signed-rank tests across seeds with Holm
correction, the teaser figure, and LaTeX tables written to `paper/tables/`. Every number in the
paper is produced here or in notebooks 01-08; `paper/claims_map.md` lists which.

In [1]:
# parameters
CONFIGS = ["configs/main_addition.yaml", "configs/main_parity.yaml", "configs/main_gates.yaml"]
TASKS = ["addition", "parity"]
SPLITS = ["id_test", "ood_depth", "ood_length", "ood_both"]
MAIN_ENTRIES = ["fixed_T", "oracle_stable", "max_softmax", "entropy", "stability", "kl", "hidden_change", "max_conf_fan",
                "trm_post", "act", "ponder", "trm_joint", "v1gate", "done_post", "done_prop_post", "done_joint", "done_prop_joint"]

In [2]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [3]:
from glwm2.eval.results import configs_from_yaml, finished, curve_rows, step_acc
from glwm2.eval.analysis import (ENTRIES, BASELINES, OURS, POOLED_JOINT, CONF_RULES, entry_rows, entry_instances,
                                 summary_table, g2_check, g3_check, binned, premature_ci, length_leakage_ci)
from glwm2.eval.plotting import savefig, FAMILY_COLORS
from glwm2.utils.io import write_json
cfgs = {t: [c for f in CONFIGS for c in configs_from_yaml(PROJECT_ROOT / f) if c.task == t] for t in TASKS}
for t in TASKS:
    print(t, f"{len(finished(cfgs[t]))}/{len(cfgs[t])} runs finished, seeds:",
          sorted({c.seed for c in finished(cfgs[t])}))
rows = {t: entry_rows(cfgs[t]) for t in TASKS}

addition 14/14 runs finished, seeds: [0, 1]
parity 14/14 runs finished, seeds: [0, 1]


In [4]:
from glwm2.eval.stats import bootstrap_ci, wilcoxon_seeds, holm
def fmt(m, lo, hi, pct=True):
    s = 100 if pct else 1
    return f"{m*s:.1f} \\tiny{{[{lo*s:.1f}, {hi*s:.1f}]}}"
tables = {}
for t in TASKS:
    r = rows[t]
    lines = [r"\begin{tabular}{l" + "rr" * len(SPLITS) + "}", r"\toprule",
             " & " + " & ".join([rf"\multicolumn{{2}}{{c}}{{{s.replace('_', '-')}}}" for s in SPLITS]) + r" \\",
             " & " + " & ".join(["Acc. (\\%)", "Steps"] * len(SPLITS)) + r" \\", r"\midrule"]
    fam_prev = None
    for e in MAIN_ENTRIES:
        if e not in set(r.entry): continue
        fam = ENTRIES[e][2]
        if fam_prev is not None and fam != fam_prev: lines.append(r"\midrule")
        fam_prev = fam
        cells = []
        for s in SPLITS:
            v = r[(r.entry == e) & (r.split == s)]
            if v.empty: cells += ["--", "--"]; continue
            m, lo, hi = bootstrap_ci(v.acc.values, n_boot=2000)
            cells += [fmt(m, lo, hi), f"{v.mean_steps.mean():.1f}"]
        lines.append(ENTRIES[e][3] + " & " + " & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    tables[t] = "\n".join(lines)
    p = PAPER_DIR / "tables" / f"main_{t}.tex"; p.write_text(tables[t]); WRITTEN.append(p)
print(tables["addition"][:1500])

\begin{tabular}{lrrrrrrrr}
\toprule
 & \multicolumn{2}{c}{id-test} & \multicolumn{2}{c}{ood-depth} & \multicolumn{2}{c}{ood-length} & \multicolumn{2}{c}{ood-both} \\
 & Acc. (\%) & Steps & Acc. (\%) & Steps & Acc. (\%) & Steps & Acc. (\%) & Steps \\
\midrule
Fixed $T$ & 100.0 \tiny{[100.0, 100.0]} & 10.0 & 70.9 \tiny{[69.5, 72.3]} & 10.0 & 100.0 \tiny{[100.0, 100.0]} & 10.0 & 49.5 \tiny{[49.5, 49.5]} & 10.0 \\
Oracle (stable) & 100.0 \tiny{[100.0, 100.0]} & 2.9 & 100.0 \tiny{[100.0, 100.0]} & 4.9 & 100.0 \tiny{[100.0, 100.0]} & 4.4 & 99.6 \tiny{[99.6, 99.7]} & 40.3 \\
\midrule
Max-softmax & 100.0 \tiny{[100.0, 100.0]} & 5.1 & 100.0 \tiny{[100.0, 100.0]} & 13.5 & 100.0 \tiny{[100.0, 100.0]} & 5.5 & 99.8 \tiny{[99.8, 99.8]} & 75.6 \\
Entropy & 100.0 \tiny{[100.0, 100.0]} & 5.1 & 100.0 \tiny{[100.0, 100.0]} & 13.5 & 100.0 \tiny{[100.0, 100.0]} & 5.5 & 99.6 \tiny{[99.4, 99.8]} & 75.4 \\
$k$-step stability & 99.9 \tiny{[99.9, 100.0]} & 3.8 & 99.9 \tiny{[99.8, 100.0]} & 5.2 & 100.0 \tiny{[10

In [5]:
# paired tests: each completion-head variant vs each baseline, OOD-depth accuracy across seeds, Holm-corrected
tests = {}
for t in TASKS:
    r = rows[t][rows[t].split == "ood_depth"]
    pv = {}
    for o in [e for e in ["done_post", "done_prop_post", "done_joint", "done_prop_joint"] if e in set(r.entry)]:
        for b in [e for e in MAIN_ENTRIES if ENTRIES[e][2] in ("posthoc", "joint") and e in set(r.entry)]:
            A = r[r.entry == o].set_index("seed").acc
            B = r[r.entry == b].set_index("seed").acc
            common = A.index.intersection(B.index)
            if len(common) >= 2:
                pv[f"{o} vs {b}"] = wilcoxon_seeds(A[common].values, B[common].values)
    tests[t] = holm(pv)
    df_t = pd.DataFrame(tests[t]).T.sort_values("p")
    print(t, f"{int(df_t.reject.sum())}/{len(df_t)} comparisons significant after Holm (alpha 0.05)")
    display(df_t.head(12))

addition 0/44 comparisons significant after Holm (alpha 0.05)


,p,p_adj,reject
done_post vs stability,0.5,1.0,False
done_prop_joint vs v1gate,0.5,1.0,False
done_prop_joint vs ponder,0.5,1.0,False
done_prop_joint vs act,0.5,1.0,False
done_prop_joint vs kl,0.5,1.0,False
done_prop_joint vs stability,0.5,1.0,False
done_joint vs v1gate,0.5,1.0,False
done_joint vs ponder,0.5,1.0,False
done_joint vs act,0.5,1.0,False
done_joint vs trm_post,0.5,1.0,False


parity 0/44 comparisons significant after Holm (alpha 0.05)


,p,p_adj,reject
done_post vs max_softmax,0.5,1.0,False
done_joint vs stability,0.5,1.0,False
done_joint vs kl,0.5,1.0,False
done_joint vs hidden_change,0.5,1.0,False
done_joint vs max_conf_fan,0.5,1.0,False
done_joint vs act,0.5,1.0,False
done_joint vs ponder,0.5,1.0,False
done_joint vs trm_joint,0.5,1.0,False
done_joint vs v1gate,0.5,1.0,False
done_joint vs entropy,0.5,1.0,False


In [6]:
# premature-stop table (OOD-depth and OOD-both), with per-instance bootstrap CIs pooled over seeds
lines = [r"\begin{tabular}{l" + "rr" * len(TASKS) + "}", r"\toprule",
         " & " + " & ".join([rf"\multicolumn{{2}}{{c}}{{{t}}}" for t in TASKS]) + r" \\",
         " & " + " & ".join(["OOD-depth", "OOD-both"] * len(TASKS)) + r" \\", r"\midrule"]
inst = {(t, s): entry_instances(cfgs[t], s) for t in TASKS for s in ["ood_depth", "ood_both"]}
for e in MAIN_ENTRIES:
    if e in ("oracle_stable",): continue
    cells = []
    for t in TASKS:
        for s in ["ood_depth", "ood_both"]:
            d = inst[(t, s)]
            if e not in set(d.entry): cells.append("--"); continue
            m, lo, hi = premature_ci(d, e)
            cells.append(fmt(m, lo, hi))
    lines.append(ENTRIES[e][3] + " & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}"]
p = PAPER_DIR / "tables" / "premature.tex"; p.write_text("\n".join(lines)); WRITTEN.append(p)

In [7]:
# Fig. 1 (teaser): one deep and long addition instance; where two global convergence rules halt
# (hidden-state change: wrong; k-step stability: before the bound) vs the propagating completion head
from glwm2.train.trainer import load_model
from glwm2.data.splits import load_split
from glwm2.eval.trajectory import per_position_rho
from glwm2.eval.results import per_instance
c = next(c for c in finished(cfgs["addition"]) if c.method == "base" and c.seed == 0)
pi = per_instance(c)["per_instance"]["ood_both"]
data = load_split("addition", "ood_both")
short = data["length"].numpy() <= 96
cand = np.where((pi["hidden_change"]["correct"] == 0) & (pi["done_prop_post"]["correct"] == 1) & short)[0]
k = int(cand[np.argmax(data["depth"].numpy()[cand])]) if len(cand) else int(torch.argmax(data["depth"]))
model = load_model(c)
L = int(data["length"][k]); rho = per_position_rho("addition", {kk: v[k:k+1] for kk, v in data.items()})[0, :L+1].numpy()
T = int(data["depth"][k]) + 8
x, y = data["x"][k:k+1].cuda(), data["y"][k:k+1].cuda()
with torch.no_grad():
    pc = torch.stack([(o["logits"].argmax(-1) == y)[0, :L+1] for o in model.rollout(x, T)]).cpu().numpy()
fig, ax = plt.subplots(figsize=(3.5, 2.4))
ax.imshow(pc.T, aspect="auto", origin="lower", cmap="Greys_r", interpolation="nearest", extent=[0.5, T + 0.5, -0.5, L + 0.5], alpha=0.85)
ax.plot(rho, np.arange(L + 1), color="#D55E00", lw=1.2, label=r"carry arrives ($\rho_i$)")
ax.axvline(pi["hidden_change"]["steps"][k], color="#CC79A7", lw=1.5, ls="--", label="hidden-change halts (wrong)")
ax.axvline(pi["stability"]["steps"][k], color="#56B4E9", lw=1.5, ls=":", label="stability halts")
ax.axvline(pi["done_prop_post"]["steps"][k], color="#009E73", lw=1.5, label="completion head halts")
ax.set_xlabel("step t"); ax.set_ylabel("output digit i"); ax.legend(fontsize=5.5, frameon=True, loc="lower right")
ax.set_title(f"{L}-digit addition, $d^*$ = {int(data['depth'][k])} (white = correct)", fontsize=7)
WRITTEN.append(savefig(fig, "fig1_teaser"))

In [8]:
from glwm2.train.trainer import run_dirs
from glwm2.utils.io import read_json
comp = []
for f in sorted((PROJECT_ROOT / "configs").glob("*.yaml")):
    if f.stem.startswith("_"): continue
    cs = finished(configs_from_yaml(f))
    tr = sum(read_json(run_dirs(c)[1] / "train_summary.json").get("train_seconds", 0) for c in cs if (run_dirs(c)[1] / "train_summary.json").exists())
    ev = sum(read_json(run_dirs(c)[1] / "eval_summary.json").get("eval_seconds", 0) for c in cs if (run_dirs(c)[1] / "eval_summary.json").exists())
    if cs: comp.append(dict(sweep=f.stem, runs=len(cs), train_h=tr / 3600, eval_h=ev / 3600))
comp = pd.DataFrame(comp)
lines = [r"\begin{table}[h]", r"\centering", r"\caption{Measured GPU time per sweep (RTX 5070; several runs shared the GPU, so wall-clock per run is higher than in isolation). Runs listed in more than one sweep are counted once per sweep.}",
         r"\label{tab:compute}", r"\small", r"\begin{tabular}{lrrr}", r"\toprule", r"Sweep & Runs & Train (h) & Eval (h) \\", r"\midrule"]
for r_ in comp.itertuples():
    lines.append(f"\\texttt{{{r_.sweep.replace('_', '\\_')}}} & {r_.runs} & {r_.train_h:.1f} & {r_.eval_h:.1f} \\\\")
lines += [r"\midrule", f"Total & {int(comp.runs.sum())} & {comp.train_h.sum():.1f} & {comp.eval_h.sum():.1f} \\\\", r"\bottomrule", r"\end{tabular}", r"\end{table}"]
p = PAPER_DIR / "tables" / "compute.tex"; p.write_text("\n".join(lines)); WRITTEN.append(p)
display(comp.round(2))
summary = {t: rows[t].groupby(["entry", "split"])[["acc", "mean_steps", "premature", "rho_depth", "rho_length_given_depth"]]
           .mean().reset_index().to_dict(orient="records") for t in TASKS}
WRITTEN.append(write_json(RESULTS_DIR / "paper_numbers.json", {"summary": summary,
               "holm": {t: tests[t] for t in TASKS}}))

,sweep,runs,train_h,eval_h
0,ablations,3,0.59,0.24
1,main_addition,8,2.57,0.65
2,main_gates,12,5.23,1.27
3,main_parity,8,2.43,1.26
4,pilot_addition,8,2.36,1.11
5,tune_gates,2,1.10,0.24


In [9]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))

Files written by this notebook:
  paper\tables\main_addition.tex
  paper\tables\main_parity.tex
  paper\tables\premature.tex
  paper\figures\fig1_teaser.pdf
  paper\tables\compute.tex
  results\paper_numbers.json
